In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
DEVICE=torch.device("cuda")

In [ ]:
DATA_PATH = "mnist_test_seq.npy"
raw = np.load(DATA_PATH, mmap_mode="r", allow_pickle=False)
assert raw.shape == (20, 10000, 64, 64)
videos = raw.transpose(1, 0, 2, 3)[:, :, None, :, :]
videos = videos.astype(np.float32) / 255.0
videos = torch.from_numpy(videos)
assert videos.shape == (10000, 20, 1, 64, 64)
assert videos.dtype == torch.float32
assert 0.0 <= videos.min() <= videos.max() <= 1.0
train_videos = videos[:8000]
val_videos = videos[8000:9000]
test_videos = videos[9000:10000]
print(train_videos.shape, val_videos.shape, test_videos.shape)

In [ ]:
import cv2
import torchvision.transforms as T
from pathlib import Path

class UCFClipDataset(Dataset):
  def __init__(self, root, class_names, n_frames=16, size=112):
    self.root        = Path(root)
    self.class_names = class_names
    self.n_frames    = n_frames
    self.size        = size
    self.label_map   = {c: i for i, c in enumerate(class_names)}
    self.clips = []
    for cls in class_names:
      vids = sorted((self.root / cls).glob("*.avi")) + sorted((self.root / cls).glob("*.mp4"))
      for v in vids:
        self.clips.append((v, self.label_map[cls]))

    self.transform = T.Compose([
        T.Resize((size, size), antialias=True),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

  def _load_clip(self, path):
    cap    = cv2.VideoCapture(str(path))
    total  = max(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), 1)
    indices = np.linspace(0, total - 1, self.n_frames, dtype=int)
    frames = []
    for idx in indices:
      cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
      ret, frame = cap.read()
      if not ret or frame is None:
        frame = np.zeros((self.size, self.size, 3), dtype=np.uint8)
      else:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (self.size, self.size))
      frames.append(frame)
    cap.release()

    arr = torch.from_numpy(np.stack(frames)).float() / 255.0
    arr = arr.permute(3, 0, 1, 2)

    return torch.stack(
        [self.transform(arr[:, t]) for t in range(arr.shape[1])], dim=1
    )
  def __len__(self):              return len(self.clips)
  def __getitem__(self, idx):
    path, label = self.clips[idx]
    return self._load_clip(path), label

N_FRAMES  = 16
CLIP_SIZE = 112
RAW_DIR   = Path("./UCF101_subset")

CLASS_NAMES = sorted([d.name for d in (RAW_DIR / "train").iterdir() if d.is_dir()])

train_ds = UCFClipDataset(RAW_DIR / "train", CLASS_NAMES, N_FRAMES, CLIP_SIZE)
test_ds  = UCFClipDataset(RAW_DIR / "test",  CLASS_NAMES, N_FRAMES, CLIP_SIZE)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False, pin_memory=True)

print(f"Train clips : {len(train_ds)}")
print(f"Test  clips : {len(test_ds)}")
print(f"Clip shape  : {train_ds[0][0].shape}  →  (C, T, H, W)")
print(f"Batch shape : {next(iter(train_loader))[0].shape}")

In [ ]:
class MovingMNISTDataset(Dataset):
    def __init__(self, videos, observed_steps=10, target_steps=10):
        self.videos = videos
        self.observed_steps = observed_steps
        self.target_steps = target_steps
    def __len__(self):
        return len(self.videos)
    def __getitem__(self, index):
        seq = self.videos[index]
        return (
            seq[:self.observed_steps],
            seq[self.observed_steps:self.observed_steps + self.target_steps]
        )
BATCH_SIZE = 16
train_loader = DataLoader(
    MovingMNISTDataset(train_videos), batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    MovingMNISTDataset(val_videos), batch_size=BATCH_SIZE, shuffle=False
)
test_loader = DataLoader(
    MovingMNISTDataset(test_videos), batch_size=BATCH_SIZE, shuffle=False
)
X, y = next(iter(train_loader))
print(X.shape, y.shape)

In [ ]:
X, y = next(iter(train_loader))
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for t in range(10):
    axes[0, t].imshow(X[0, t, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, t].imshow(y[0, t, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, t].axis("off")
    axes[1, t].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super().__init__()
        if kernel_size % 2 != 1:
            raise ValueError("kernel_size must be odd")
        self.hidden_channels = hidden_channels
        padding = kernel_size // 2
        self.gate_conv = nn.Conv2d(
            in_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size=kernel_size,
            padding=padding,
        )
    def forward(self, x, state=None):
        B, _, H, W = x.shape
        if state is None:
            h_prev = x.new_zeros(
                B, self.hidden_channels, H, W
            )
            c_prev = x.new_zeros(
                B, self.hidden_channels, H, W
            )
        else:
            h_prev, c_prev = state
        combined = torch.cat((x, h_prev), dim=1)
        packed = self.gate_conv(combined)
        i, f, o, g = packed.chunk(4, dim=1)
        i = torch.sigmoid(i)   # input gate
        f = torch.sigmoid(f)   # forget gate
        o = torch.sigmoid(o)   # output gate
        g = torch.tanh(g)      # candidate
        c = f * c_prev + i * g
        h = o * torch.tanh(c)
        return h, c
class ConvLSTMEncoder(nn.Module):
    def __init__(self, hidden_channels=(64, 64), kernel_size=3):
        super().__init__()
        self.cells = nn.ModuleList([
            ConvLSTMCell(
                in_channels=1,
                hidden_channels=hidden_channels[0],
                kernel_size=kernel_size,
            ),
            ConvLSTMCell(
                in_channels=hidden_channels[0],
                hidden_channels=hidden_channels[1],
                kernel_size=kernel_size,
            ),
        ])
    def forward(self, x):
        """
        x:
            (B, T, 1, H, W)
        Returns:
            states = [
                (h1, c1),
                (h2, c2)
            ]
        """
        states = [None, None]
        for t in range(x.shape[1]):
            xt = x[:, t]
            states[0] = self.cells[0](
                xt,
                states[0],
            )
            h1, _ = states[0]
            states[1] = self.cells[1](
                h1,
                states[1],
            )
        return states
class ConvLSTMForecaster(nn.Module):
    def __init__(self, hidden_channels=(64, 64), kernel_size=3):
        super().__init__()
        self.cells = nn.ModuleList([
            ConvLSTMCell(
                in_channels=1,
                hidden_channels=hidden_channels[0],
                kernel_size=kernel_size,
            ),
            ConvLSTMCell(
                in_channels=hidden_channels[0],
                hidden_channels=hidden_channels[1],
                kernel_size=kernel_size,
            ),
        ])
        self.output_head = nn.Conv2d(
            hidden_channels[1],
            1,
            kernel_size=1,
        )
    def forward(self, states, last_observed, targets=None, n_steps=10, tf_ratio=0.0):
        """
        states:
            Encoder states:
            [
                (h1, c1),
                (h2, c2)
            ]

        last_observed:
            Last observed frame.
            Shape: (B, 1, H, W)

        targets:
            Ground-truth future frames.
            Shape: (B, T, 1, H, W)

            Required during training when tf_ratio > 0.

        n_steps:
            Number of future frames to predict.

        tf_ratio:
            1.0 -> always teacher forcing
            0.0 -> never teacher forcing
        """
        h1, c1 = states[0]
        h2, c2 = states[1]
        # First decoder input is the last observed frame
        current = last_observed
        predictions = []
        for t in range(n_steps):
            h1, c1 = self.cells[0](
                current,
                (h1, c1),
            )
            h2, c2 = self.cells[1](
                h1,
                (h2, c2),
            )
            prediction = torch.sigmoid(
                self.output_head(h2)
            )
            predictions.append(prediction)
            if targets is not None and t < targets.shape[1]:
                if tf_ratio > 0:
                    use_teacher = (
                        torch.rand(
                            1,
                            device=prediction.device,
                        ).item()
                        < tf_ratio
                    )
                    if use_teacher:
                        current = targets[:, t]
                    else:
                        current = prediction
                else:
                    current = prediction
            else:
                current = prediction
        return torch.stack(predictions, dim=1)
class ConvLSTMModel(nn.Module):
    def __init__(self, hidden_channels=(64, 64), kernel_size=3):
        super().__init__()
        self.encoder = ConvLSTMEncoder(
            hidden_channels=hidden_channels,
            kernel_size=kernel_size,
        )
        self.forecaster = ConvLSTMForecaster(
            hidden_channels=hidden_channels,
            kernel_size=kernel_size,
        )
    def forward(self, x, targets=None, n_steps=10, tf_ratio=0.0):
        """
        x:
            Observed frames
            (B, T_obs, 1, H, W)

        targets:
            Future ground-truth frames
            (B, T_target, 1, H, W)

        Returns:
            predictions
            (B, n_steps, 1, H, W)
        """
        states = self.encoder(x)
        predictions = self.forecaster(
            states=states,
            last_observed=x[:, -1],
            targets=targets,
            n_steps=n_steps,
            tf_ratio=tf_ratio,
        )
        return predictions

In [ ]:
class ConvGRUCell(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super().__init__()
        if kernel_size % 2 != 1:
            raise ValueError("kernel_size must be odd")
        self.hidden_channels = hidden_channels
        padding = kernel_size // 2
        self.xgate = nn.Conv2d(
            in_channels, 3 * hidden_channels,
            kernel_size=kernel_size, padding=padding
        )
        self.hgate = nn.Conv2d(
            hidden_channels, 3 * hidden_channels,
            kernel_size=kernel_size, padding=padding
        )
    def forward(self, x, h=None):
        B, _, H, W = x.shape
        if h is None:
            h = x.new_zeros(B, self.hidden_channels, H, W)
        xz, xr, xh = self.xgate(x).chunk(3, dim=1)
        hz, hr, hh = self.hgate(h).chunk(3, dim=1)
        z = torch.sigmoid(xz + hz)
        r = torch.sigmoid(xr + hr)
        # NOTE: Handbook uses tanh while paper uses LeakyRelu(with 0.2)
        h_tilde = torch.tanh(xh + r * hh)
        return (1 - z) * h_tilde + z * h

In [ ]:
def warp(feature, u, v):
    B, C, H, W = feature.shape
    device, dtype = feature.device, feature.dtype
    y = (torch.arange(H, device=device, dtype=dtype) + 0.5) * (2 / H) - 1
    x = (torch.arange(W, device=device, dtype=dtype) + 0.5) * (2 / W) - 1
    yy, xx = torch.meshgrid(y, x, indexing="ij")
    base = torch.stack((xx, yy), dim=-1)
    flow_norm = torch.stack(
        (u * (2 / W), v * (2 / H)), dim=-1
    )
    grid = base.unsqueeze(0) + flow_norm
    return F.grid_sample(
        feature,
        grid,
        mode="bilinear",
        padding_mode="zeros",
        align_corners=False,
    )
feature = torch.randn(2, 4, 8, 8, device=DEVICE, requires_grad=True)
zero_u = torch.zeros(2, 8, 8, device=DEVICE, requires_grad=True)
zero_v = torch.zeros(2, 8, 8, device=DEVICE, requires_grad=True)
warped = warp(feature, zero_u, zero_v)
assert torch.allclose(warped, feature, atol=1e-6)
loss = warped.square().mean()
loss.backward()
print("zero-flow error:", (warped - feature).abs().max().item())

In [ ]:
class FlowSubnetwork(nn.Module):
    def __init__(self, in_channels, hidden_channels, L=13):
        super().__init__()
        self.L = L
        self.net = nn.Sequential(
            nn.Conv2d(
                in_channels + hidden_channels,
                32,
                kernel_size=5,
                padding=2,
            ),
            nn.LeakyReLU(negative_slope=0.2),
            nn.Conv2d(
                32,
                2 * L,
                kernel_size=5,
                padding=2,
            ),
        )
    def forward(self, x, h):
        flow = self.net(torch.cat((x, h), dim=1))
        B, _, H, W = flow.shape
        flow = flow.reshape(B, self.L, 2, H, W)
        return flow[:, :, 0], flow[:, :, 1]
class TrajGRUCell(nn.Module):
    def __init__(self, in_channels, hidden_channels, kernel_size=3, L=13):
        super().__init__()
        if kernel_size % 2 != 1:
            raise ValueError("kernel_size must be odd")
        self.hidden_channels = hidden_channels
        self.L = L
        padding = kernel_size // 2
        self.flow_net = FlowSubnetwork(in_channels, hidden_channels, L=L)
        self.xgate = nn.Conv2d(
            in_channels,
            3 * hidden_channels,
            kernel_size=kernel_size,
            padding=padding,
        )
        self.h_proj = nn.ModuleList([
            nn.Conv2d(hidden_channels, 3 * hidden_channels, kernel_size=1)
            for _ in range(L)
        ])
    def forward(self, x, h=None):
        B, _, H, W = x.shape
        if h is None:
            h = x.new_zeros(B, self.hidden_channels, H, W)
        U, V = self.flow_net(x, h)
        projected = 0
        for l in range(self.L):
            warped = warp(h, U[:, l], V[:, l])
            projected = projected + self.h_proj[l](warped)
        xz, xr, xh = self.xgate(x).chunk(3, dim=1)
        hz, hr, hh = projected.chunk(3, dim=1)
        z = torch.sigmoid(xz + hz)
        r = torch.sigmoid(xr + hr)
        # NOTE: see ConvGRUCell -- tanh per handbook 5.3, not the paper's
        h_tilde = torch.tanh(xh + r * hh)
        return (1 - z) * h_tilde + z * h
class TrajGRUEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.cells = nn.ModuleList([
            TrajGRUCell(1, 64, L=13),
            TrajGRUCell(64, 64, L=13),
        ])
    def forward(self, x):
        h1 = None
        h2 = None
        for t in range(x.shape[1]):
            xt = x[:, t]
            h1 = self.cells[0](xt, h1)
            h2 = self.cells[1](h1, h2)
        return [h1, h2]
class TrajGRUForecaster(nn.Module):
    def __init__(self):
        super().__init__()
        self.cells = nn.ModuleList([
            TrajGRUCell(1, 64, L=13),
            TrajGRUCell(64, 64, L=13),
        ])
        self.output_head = nn.Conv2d(64, 1, kernel_size=1)
    def forward(self, states, n_steps=10):
        h1, h2 = states
        current = h1.new_zeros(h1.shape[0], 1, h1.shape[2], h1.shape[3])
        predictions = []
        for _ in range(n_steps):
            h1 = self.cells[0](current, h1)
            h2 = self.cells[1](h1, h2)
            current = torch.sigmoid(self.output_head(h2))
            predictions.append(current)
        return torch.stack(predictions, dim=1)
class TrajGRUModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = TrajGRUEncoder()
        self.forecaster = TrajGRUForecaster()
    def forward(self, x, n_steps=10):
        states = self.encoder(x)
        return self.forecaster(states, n_steps)

In [ ]:
X, y = next(iter(train_loader))
X, y = X.to(DEVICE), y.to(DEVICE)
convlstm = ConvLSTMModel().to(DEVICE)
trajgru = TrajGRUModel().to(DEVICE)
with torch.no_grad():
    pred_c = convlstm(X)
    pred_t = trajgru(X)
assert pred_c.shape == pred_t.shape == y.shape
print(pred_c.shape, pred_t.shape)
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_samples = 0
    for X, y in tqdm(loader, leave=False):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * X.size(0)
        total_samples += X.size(0)
    return total_loss / total_samples
@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    for X, y in tqdm(loader, leave=False):
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = criterion(pred, y)
        total_loss += loss.item() * X.size(0)
        total_samples += X.size(0)
    return total_loss / total_samples
@torch.no_grad()
def per_frame_mse(model, loader, device):
    model.eval()
    error_sum = None
    count = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        err = (pred - y).square().sum(dim=(0, 2, 3, 4))
        if error_sum is None:
            error_sum = torch.zeros_like(err)
        error_sum += err
        count += X.shape[0] * y.shape[2] * y.shape[3] * y.shape[4]
    return (error_sum / count).cpu()
# mse_c = per_frame_mse(convlstm, test_loader, DEVICE)
# mse_t = per_frame_mse(trajgru, test_loader, DEVICE)
@torch.no_grad()
def show_forecast_comparison(convlstm, trajgru, loader, device, sample_idx):
    convlstm.eval()
    trajgru.eval()
    X, y = next(iter(loader))
    X, y = X.to(device), y.to(device)
    pred_c = convlstm(X)
    pred_t = trajgru(X)
    frames = [0, 4, 9]
    names = ["Ground Truth", "ConvLSTM", "TrajGRU"]
    fig, axes = plt.subplots(3, 3, figsize=(7, 7))
    for row, (name, seq) in enumerate([
        (names[0], y),
        (names[1], pred_c),
        (names[2], pred_t),
    ]):
        for col, t in enumerate(frames):
            axes[row, col].imshow(
                seq[sample_idx, t, 0].cpu(),
                cmap="gray",
                vmin=0,
                vmax=1,
            )
            axes[row, col].axis("off")
            if row == 0:
                axes[row, col].set_title(f"Frame {t+1}")
            if col == 0:
                axes[row, col].set_ylabel(name)
    plt.tight_layout()
    plt.show()

In [ ]:
def check_zero_flow():
    feature = torch.randn(2, 3, 8, 8, device=DEVICE)
    u = torch.zeros(2, 8, 8, device=DEVICE)
    v = torch.zeros(2, 8, 8, device=DEVICE)
    out = warp(feature, u, v)
    assert torch.allclose(out, feature, atol=1e-6)
check_zero_flow()
print("zero-flow test passed")
def csi_hss(pred, target, threshold=0.5):
    pred_bin = (pred >= threshold).float()
    target_bin = (target >= threshold).float()
    tp = (pred_bin * target_bin).sum()
    fn = ((1 - pred_bin) * target_bin).sum()
    fp = (pred_bin * (1 - target_bin)).sum()
    tn = ((1 - pred_bin) * (1 - target_bin)).sum()
    csi = tp / (tp + fn + fp + 1e-8)
    hss = (tp * tn - fn * fp) / (
        (tp + fn) * (fn + tn) +
        (tp + fp) * (fp + tn) +
        1e-8
    )
    return csi.item(), hss.item()
def rainfall_weight(x):
    w = torch.ones_like(x)
    w = torch.where(x >= 2,  torch.full_like(x, 2.0),  w)
    w = torch.where(x >= 5,  torch.full_like(x, 5.0),  w)
    w = torch.where(x >= 10, torch.full_like(x, 10.0), w)
    w = torch.where(x >= 30, torch.full_like(x, 30.0), w)
    return w
def b_mse_b_mae(pred, target):
    w = rainfall_weight(target)
    b_mse = (w * (pred - target).square()).mean()
    b_mae = (w * (pred - target).abs()).mean()
    return b_mse.item(), b_mae.item()